# Procesamiento de señales intracraneales (ECoG) y modelado computacional con Python
### Taller de neuroingeniería clínica · Sistemas Neuroelectrónicos

En este taller vamos a **replicar, con datos reales de pacientes**, los hallazgos centrales de
Miller et al. (2010): qué ocurre en la superficie de la corteza motora cuando una persona mueve la mano
o la lengua, y qué ocurre cuando solo *imagina* ese movimiento. Esa es la base fisiológica de una BCI
basada en ECoG.

**Al terminar podrás:**
1. Cargar y explorar registros ECoG clínicos (matriz *tiempo × canales*).
2. Estimar y modelar el espectro de potencia (componente 1/f + oscilaciones) con *specparam*.
3. Extraer la envolvente de alta frecuencia (HFB) con filtrado pasa-banda y transformada de *Hilbert*.
4. Construir mapas de activación funcional y compararlos con los del artículo.
5. Entrenar y evaluar un decodificador, y reconocer el *data leakage*.

| Bloque | Tiempo aprox. |
|---|---|
| 0. Configuración | 5 min |
| 1. Los datos | 15 min |
| 2. Espectro y modelo 1/f | 30 min |
| ☕ Descanso | 10 min |
| 3. Envolvente de alta frecuencia | 35 min |
| 4. Mapeo funcional | 20 min |
| 5. Decodificación y BCI | 35 min |
| 6. Extra: movimientos de dedos | 20 min |
| Apéndice: replicación grupal (opcional) | 1 min |

**Cómo usar este notebook.** Ejecuta las celdas **en orden**, una por una (▶ o `Shift + Enter`).
Las celdas con título y código oculto son plomería: solo ejecútalas.

| Icono | Significado |
|---|---|
| 🧠 | Concepto clave |
| 🎯 | Objetivo de réplica: una figura del artículo que vamos a reproducir |
| ✍️ | **TODO**: te toca completar 1 o 2 líneas |
| ✔️ | Celda de verificación: te dice si tu TODO quedó bien (y, si no, te deja continuar) |
| 💡 | Pregunta para pensar o discutir |

> **Datos:** Miller, K. J. (2019). *A library of human electrocorticographic data and analyses*.
> Nature Human Behaviour, 3(11), 1225–1235. Versión curada por Neuromatch Academy
> (Marius Pachitariu y equipo de TAs). Las figuras se reproducen con fines educativos y
> no comerciales, citando su fuente.

## 0. Configuración

Ejecuta las cuatro celdas siguientes. La instalación tarda menos de un minuto.

In [ ]:
#@title 📦 Instalar paquetes
# Versiones fijadas para que el taller sea reproducible.
# nilearn 0.13.1 y no una más nueva: desde la 0.14 exige pandas >= 2.3 y requests >= 2.33, que chocan
# con las versiones que Colab fija (pandas 2.2.3, requests 2.32.4) y obligan a reiniciar el entorno.
# specparam 2.0 todavía es "release candidate": si cambias la versión, revisa la sección 2.
%pip install -q nilearn==0.13.1 specparam==2.0.0rc7

In [ ]:
#@title 📚 Importar bibliotecas y estilo de gráficas
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import requests
from scipy import fft as sp_fft
from scipy import signal, stats

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({
    "figure.figsize": (12, 4), "font.size": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.autolayout": True,
})
C_HAND, C_TONGUE, C_REST = "#d62728", "#1f77b4", "#555555"
C_LFB, C_HFB = "#2ca02c", "#ff9f1c"
print("Listo ✅")

In [ ]:
#@title 🔧 Funciones auxiliares (no necesitas modificarlas)
REPO_URL = "https://github.com/EmmanuelIsaiasGuizarBayardo/taller-ecog"
RELEASE_TAG = "data-v1"
DATA_SOURCES = {
    "motor_imagery": {
        "local": "motor_imagery.npz",
        "urls": [f"{REPO_URL}/releases/download/{RELEASE_TAG}/motor_imagery.npz",
                 "https://osf.io/ksqv8/download"],
    },
    "fingerflex": {
        "local": "fingerflex.npz",
        "urls": [f"{REPO_URL}/releases/download/{RELEASE_TAG}/fingerflex.npz",
                 "https://osf.io/5m47z/download"],
    },
}


def load_npz_dataset(name: str) -> np.ndarray:
    """Carga un dataset de NMA probando fuentes en cadena.

    Orden: archivo local -> GitHub Release -> OSF.

    Parameters
    ----------
    name : str
        Clave en ``DATA_SOURCES`` ("motor_imagery" o "fingerflex").

    Returns
    -------
    alldat : np.ndarray
        Arreglo de objetos (sujetos x bloques) con un ``dict`` por registro.
    """
    cfg = DATA_SOURCES[name]
    fname = cfg["local"]
    if not os.path.isfile(fname):
        for url in cfg["urls"]:
            try:
                print(f"Descargando {name} desde {url} ...")
                with requests.get(url, stream=True, timeout=120) as r:
                    r.raise_for_status()
                    with open(fname, "wb") as fid:
                        for chunk in r.iter_content(chunk_size=1 << 20):
                            fid.write(chunk)
                break
            except Exception as err:  # noqa: BLE001
                print(f"  ⚠️ Falló ({err}); probando la siguiente fuente.")
                if os.path.exists(fname):
                    os.remove(fname)
        else:
            raise RuntimeError("No se pudo descargar. Sube el .npz manualmente al panel de archivos.")
    return np.load(fname, allow_pickle=True)["dat"]


# Transformación Talairach -> MNI (inversa de icbm_spm2tal; Lancaster et al., 2007).
_ICBM_SPM2TAL = np.array([[0.9254, 0.0024, -0.0118, -1.0207],
                          [-0.0048, 0.9316, -0.0871, -1.7667],
                          [0.0152, 0.0883, 0.8924, 4.0926],
                          [0.0, 0.0, 0.0, 1.0]])


def tal2mni(xyz: np.ndarray) -> np.ndarray:
    """Convierte coordenadas Talairach (n, 3) a MNI (n, 3)."""
    homog = np.c_[xyz, np.ones(len(xyz))]
    return (np.linalg.inv(_ICBM_SPM2TAL) @ homog.T).T[:, :3]


def plot_brain(values: np.ndarray, locs_mni: np.ndarray, title: str = "",
               ax: plt.Axes | None = None, vmax: float | None = None,
               cmap: str = "RdBu_r") -> None:
    """Dibuja valores por electrodo sobre un cerebro de vidrio (vista lateral).

    El tamaño y el color de cada punto codifican el valor; la escala es simétrica.
    """
    from nilearn import plotting
    values = np.asarray(values, dtype=float)
    hemi = "l" if np.mean(locs_mni[:, 0]) < 0 else "r"
    vmax = vmax or float(np.nanmax(np.abs(values))) or 1.0
    sizes = 15 + 160 * np.abs(values) / vmax
    plotting.plot_markers(values, locs_mni, node_size=sizes, node_cmap=cmap,
                          node_vmin=-vmax, node_vmax=vmax, display_mode=hemi,
                          axes=ax, title=title, colorbar=True, annotate=False)


def trial_segments(t_on: np.ndarray, t_off: np.ndarray, n_samples: int,
                   skip: int = 500) -> tuple[list, list, np.ndarray]:
    """Ventanas de movimiento (cue) y de reposo (entre cues), en muestras.

    Se descartan los primeros ``skip`` ms tras cada transición para evitar el
    tiempo de reacción y los transitorios. El reposo se recorta a la longitud
    típica de una ventana de movimiento para comparar ventanas equivalentes.

    Returns
    -------
    move : list of (inicio, fin), una por instrucción
    rest : list of (inicio, fin), el reposo que sigue a cada instrucción
    rest_from : np.ndarray, índice de la instrucción que precede a cada reposo
    """
    move = [(a + skip, b) for a, b in zip(t_on, t_off)]
    max_len = int(np.median([b - a for a, b in move]))
    nxt = list(t_on[1:]) + [n_samples]
    rest, rest_from = [], []
    for k, (b, c) in enumerate(zip(t_off, nxt)):
        end = min(c, b + skip + max_len)
        if end - (b + skip) >= 1000:
            rest.append((b + skip, end))
            rest_from.append(k)
    return move, rest, np.array(rest_from)


def segment_psds(V: np.ndarray, segments: list, fs: int,
                 fmax: float = 200.0) -> tuple[np.ndarray, np.ndarray]:
    """PSD de Welch por segmento.

    Returns
    -------
    freqs : (n_freqs,)
    psd : (n_segments, n_channels, n_freqs)
    """
    out = []
    for a, b in segments:
        f, p = signal.welch(V[a:b], fs=fs, nperseg=fs, noverlap=fs // 2, axis=0)
        out.append(p.T)
    keep = f <= fmax
    return f[keep], np.stack(out)[:, :, keep]


def band_power(psd: np.ndarray, freqs: np.ndarray, band: tuple[float, float]) -> np.ndarray:
    """Potencia media dentro de una banda: (n_seg, n_ch, n_f) -> (n_seg, n_ch)."""
    mask = (freqs >= band[0]) & (freqs <= band[1])
    return psd[..., mask].mean(axis=-1)


def band_envelope(V: np.ndarray, fs: int, band: tuple[float, float],
                  smooth_hz: float | None = None, order: int = 4,
                  chunk: int = 8) -> np.ndarray:
    """Potencia instantánea en una banda: pasa-banda -> Hilbert -> |.|^2.

    Usa SOS + ``sosfiltfilt`` (fase cero y estabilidad numérica) y rellena la
    FFT de Hilbert a una longitud "rápida" para no depender de N primo.
    """
    sos = signal.butter(order, band, btype="bandpass", fs=fs, output="sos")
    sos_lp = signal.butter(2, smooth_hz, btype="lowpass", fs=fs, output="sos") if smooth_hz else None
    n = V.shape[0]
    n_fft = sp_fft.next_fast_len(n)
    out = np.empty(V.shape, dtype=np.float32)
    # Procesamos por bloques de canales: la señal analítica es compleja de 128 bits y,
    # con registros largos, calcularla para todos los canales a la vez agota la RAM.
    for start in range(0, V.shape[1], chunk):
        sl = slice(start, start + chunk)
        x = signal.sosfiltfilt(sos, V[:, sl], axis=0)
        power = np.abs(signal.hilbert(x, N=n_fft, axis=0)[:n]) ** 2
        if sos_lp is not None:
            power = signal.sosfiltfilt(sos_lp, power, axis=0)
            # El suavizado puede dejar valores <= 0. Si los acotáramos a un número diminuto, su
            # logaritmo sería enorme y negativo (p. ej., -87) y arruinaría promedios y correlaciones.
            # Usamos un piso relativo: 0.1% de la mediana de cada canal.
            power = np.maximum(power, 1e-3 * np.median(power, axis=0))
        out[:, sl] = power
    return out


def epoch(X: np.ndarray, onsets: np.ndarray, tmin_ms: int, tmax_ms: int,
          fs: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Corta épocas alrededor de ``onsets``.

    Returns
    -------
    epochs : (n_ep, n_times, n_ch)
    times_ms : (n_times,)
    keep : (n_onsets,) bool, qué eventos cupieron completos dentro del registro
    """
    lags = np.arange(int(tmin_ms * fs / 1000), int(tmax_ms * fs / 1000))
    keep = (onsets + lags[0] >= 0) & (onsets + lags[-1] < len(X))
    return X[onsets[keep][:, None] + lags[None, :]], lags / fs * 1000, keep


def load_block(dat: dict, keep: np.ndarray | None = None) -> np.ndarray:
    """Voltaje de un bloque en float32 y µV (deshace el z-score con ``scale_uv``)."""
    V = dat["V"].astype("float32")
    if "scale_uv" in dat:
        V *= np.asarray(dat["scale_uv"], dtype="float32")
    return V if keep is None else V[:, keep]


def verify(name: str, reference, hint: str = "", rtol: float = 1e-3) -> None:
    """Compara la variable ``name`` del alumno contra la referencia.

    Si no coincide (o no existe), avisa y la reemplaza por la referencia
    para que el resto del notebook siga funcionando.
    """
    value = globals().get(name, None)
    try:
        ok = value is not None and value is not Ellipsis and np.allclose(
            np.asarray(value, dtype=float), np.asarray(reference, dtype=float),
            rtol=rtol, equal_nan=True)
    except Exception:  # noqa: BLE001
        ok = False
    if ok:
        print(f"✅ {name}: ¡correcto!")
    else:
        print(f"❌ {name}: todavía no coincide. {hint}")
        print("   Usamos el valor de referencia para que puedas continuar.")
        globals()[name] = reference


print("Funciones listas ✅")

In [ ]:
#@title 💾 Descargar los datos de *motor imagery*
alldat = load_npz_dataset("motor_imagery")
print("Sujetos disponibles:", alldat.shape[0], "| bloques por sujeto:", alldat.shape[1])

## 1. Los datos

Cada paciente tenía una rejilla de electrodos sobre la corteza, colocada para localizar el foco de
su epilepsia. Hizo dos bloques del mismo experimento:

- **Bloque 0, movimiento real:** abrir y cerrar la mano, o sacar y meter la lengua, durante ~3 s;
  luego ~3 s de reposo.
- **Bloque 1, imaginación:** lo mismo, pero solo *imaginando* el movimiento (sensación kinestésica).

Cada registro es un diccionario de Python con estas llaves:

| Llave | Contenido |
|---|---|
| `V` | Voltaje, matriz de **muestras × canales** (ya con *notch* y *z-score*) |
| `srate` | Frecuencia de muestreo (1000 Hz) |
| `t_on`, `t_off` | Inicio y fin de cada instrucción, en muestras |
| `stim_id` | Tipo de instrucción: **11 = lengua**, **12 = mano** |
| `locs` | Posición 3D de cada electrodo (Talairach) |

In [ ]:
SUBJECT = 1  #@param {type:"slider", min:0, max:6, step:1}

SUBJECT = min(SUBJECT, alldat.shape[0] - 1)   # por si el archivo trae menos sujetos
dat_real = alldat[SUBJECT][0]   # bloque de movimiento real
dat_imag = alldat[SUBJECT][1]   # bloque de imaginación
srate = int(dat_real["srate"])
print("Llaves:", list(dat_real.keys()))

🧠 **Mini repaso de NumPy.** Una matriz tiene una *forma* (`shape`). Para `V`, la forma es
`(n_muestras, n_canales)`: cada **fila** es un instante de tiempo y cada **columna** es un electrodo.

- `V[:, 5]` → todo el tiempo del canal 5.
- `V[1000:2000, :]` → del segundo 1 al 2 del registro, en todos los canales (a 1000 Hz).
- `V.mean(axis=0)` → promedio a lo largo del tiempo (uno por canal).
- `V.mean(axis=1)` → promedio a lo largo de los canales (uno por instante).

🧠 **De vuelta a microvoltios.** Para que el archivo pese menos, los curadores guardaron cada canal
normalizado (*z-score*) y en `float16`. La llave `scale_uv` guarda la desviación estándar original
de cada canal, así que multiplicar por ella recupera el voltaje en **µV**. Además descartamos
canales "planos" (con una escala muy por debajo de la mediana), que suelen ser electrodos
desconectados o de muy alta impedancia.

In [ ]:
#@title 🩺 Recuperar µV y control de calidad de canales
scale_real = np.asarray(dat_real.get("scale_uv", np.ones(dat_real["V"].shape[1])), dtype=float)
scale_imag = np.asarray(dat_imag.get("scale_uv", np.ones(dat_imag["V"].shape[1])), dtype=float)
good = (scale_real >= 0.2 * np.median(scale_real)) & (scale_imag >= 0.2 * np.median(scale_imag))
ch_ids = np.where(good)[0]                       # índice original de cada canal que conservamos
areas = np.array([f"{ba.replace('Brodmann area ', 'BA')} · {gy}"
                  for ba, gy in zip(dat_real.get("Brodmann_Area", ["?"] * len(good)),
                                    dat_real.get("gyrus", ["?"] * len(good)))])[good]
print(f"Canales conservados: {good.sum()} de {len(good)}"
      + (f" | descartados: {[int(k) for k in np.where(~good)[0]]}" if (~good).any() else ""))
print(f"Desviación estándar por canal: {scale_real[good].min():.0f} a {scale_real[good].max():.0f} µV")

In [ ]:
V = load_block(dat_real, good)   # float32, en µV, solo canales buenos

# ✍️ TODO 1: usa V.shape para obtener el número de muestras y de canales,
#            y calcula la duración del registro en segundos.
n_samples = ...
n_channels = ...
duration_s = ...

In [ ]:
#@title ✔️ Verificar TODO 1
verify("n_samples", V.shape[0], "Pista: es el primer elemento de V.shape.")
verify("n_channels", V.shape[1], "Pista: es el segundo elemento de V.shape.")
verify("duration_s", V.shape[0] / srate, "Pista: muestras ÷ muestras por segundo.")
print(f"{n_samples} muestras, {n_channels} canales, {duration_s:.1f} s")

In [ ]:
t_on, stim_id = dat_real["t_on"], dat_real["stim_id"]
t_off = dat_real.get("t_off", t_on + 3000)
print("Ensayos de mano:", np.sum(stim_id == 12), "| de lengua:", np.sum(stim_id == 11))
print("Duración típica de la instrucción:", np.median(t_off - t_on), "ms")

### ¿Dónde están los electrodos?

Las coordenadas `locs` están en espacio Talairach; las convertimos a MNI para dibujarlas sobre un
cerebro estándar. La vista es interactiva: gírala y pasa el cursor sobre los puntos para ver el
número de canal.

In [ ]:
from nilearn import plotting

locs_mni = tal2mni(dat_real["locs"][good])
plotting.view_markers(locs_mni, marker_labels=[f"{k}: {a}" for k, a in enumerate(areas)],
                      marker_color="purple", marker_size=5)

### La señal cruda

Elige un canal y mira 10 segundos de señal. Las franjas de color marcan las instrucciones.

In [ ]:
CHANNEL = 20      #@param {type:"slider", min:0, max:63, step:1}
START_S = 30      #@param {type:"slider", min:0, max:300, step:5}
CHANNEL = min(CHANNEL, n_channels - 1)

start, stop = START_S * srate, (START_S + 10) * srate
# ✍️ TODO 2: extrae de V el tramo [start:stop] del canal CHANNEL.
x = ...

In [ ]:
#@title ✔️ Verificar TODO 2 y graficar
verify("x", V[start:stop, CHANNEL], "Pista: V[filas, columna] con filas = start:stop.")
t_axis = np.arange(start, stop) / srate
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(t_axis, x, color="k", lw=0.5)
for a, b, s in zip(t_on, t_off, stim_id):
    if b > start and a < stop:
        ax.axvspan(max(a, start) / srate, min(b, stop) / srate, alpha=0.2,
                   color=C_HAND if s == 12 else C_TONGUE)
ax.set(xlabel="Tiempo (s)", ylabel="Voltaje (µV)", title=f"Canal {CHANNEL} ({areas[CHANNEL]}) · rojo: mano, azul: lengua")
plt.show()

💡 ¿Puedes ver a simple vista la diferencia entre movimiento y reposo? (Spoiler: casi nunca.
Por eso necesitamos el dominio de la frecuencia.)

### 🧠 Referencia promedio común (CAR)

Cada electrodo mide su voltaje *respecto a una referencia*. Si la referencia tiene ruido, ese ruido
aparece en **todos** los canales. La **CAR** resta a cada canal el promedio de todos los canales en
cada instante; así se elimina lo que es común a toda la rejilla y se resalta la actividad local.
Miller et al. (2010) usan precisamente esta referencia.

In [ ]:
# ✍️ TODO 3: resta a cada canal el promedio de todos los canales en cada instante.
#            Pista: V.mean(axis=1, keepdims=True) tiene forma (n_muestras, 1).
V_car = ...

In [ ]:
#@title ✔️ Verificar TODO 3
verify("V_car", V - V.mean(axis=1, keepdims=True), "Pista: V - (promedio sobre canales).")
V_imag = load_block(dat_imag, good)
V_imag_car = V_imag - V_imag.mean(axis=1, keepdims=True)
t_on_im, stim_im = dat_imag["t_on"], dat_imag["stim_id"]
t_off_im = dat_imag.get("t_off", t_on_im + 3000)
print("CAR aplicada también al bloque de imaginación ✅")

In [ ]:
#@title 🔌 Demo: ruido de línea de 60 Hz y filtro *notch*
SHOW_NOTCH_DEMO = True  #@param {type:"boolean"}
if SHOW_NOTCH_DEMO:
    seg = V_car[: 60 * srate, CHANNEL]
    tt = np.arange(seg.size) / srate
    noisy = seg + 0.5 * np.sin(2 * np.pi * 60 * tt)
    b_n, a_n = signal.iirnotch(w0=60, Q=30, fs=srate)
    clean = signal.filtfilt(b_n, a_n, noisy)
    fig, ax = plt.subplots(figsize=(10, 3.5))
    for y, lab in [(noisy, "con 60 Hz inyectado"), (clean, "después del notch")]:
        f, p = signal.welch(y, fs=srate, nperseg=srate)
        ax.semilogy(f, p, label=lab)
    ax.set(xlim=(0, 150), xlabel="Frecuencia (Hz)", ylabel="Potencia")
    ax.legend(); plt.show()

## 2. El espectro de potencia

### 🎯 Objetivo de réplica: Miller et al. (2010), Fig. 1A

<img src="https://raw.githubusercontent.com/EmmanuelIsaiasGuizarBayardo/taller-ecog/main/img/miller2010_fig1.jpg" width="420">

*Fuente: Miller et al. (2010), PNAS, Fig. 1. Uso educativo no comercial.*

En el panel A, la potencia en baja frecuencia (**LFB, 8–32 Hz**, verde) **baja** con el movimiento,
y la potencia en alta frecuencia (**HFB, 76–100 Hz**, naranja) **sube**. Vamos a reproducirlo.

🧠 **Método de Welch.** Para estimar el espectro de un segmento, lo partimos en ventanas de 1 s
que se traslapan a la mitad, calculamos la FFT de cada una y promediamos. Promediar reduce la
varianza del estimado. Con ventanas de 1 s, la resolución es de 1 Hz.

🧠 **¿Contra qué reposo comparamos?** Igual que en el artículo, cada condición se compara con el
reposo que viene **justo después** de ese tipo de instrucción. La razón: al terminar un movimiento
el ritmo beta "rebota" por encima de su nivel basal, y ese rebote depende del movimiento que se
hizo.

In [ ]:
seg_move, seg_rest, rest_from = trial_segments(t_on, t_off, n_samples)
is_hand = stim_id == 12

freqs, psd_move = segment_psds(V_car, seg_move, srate)
_, psd_rest = segment_psds(V_car, seg_rest, srate)
psd_hand, psd_tongue = psd_move[is_hand], psd_move[~is_hand]
notch = np.zeros(freqs.shape, bool)              # bins afectados por el notch de los curadores
for _c in (60, 120, 180):
    notch |= np.abs(freqs - _c) <= 3
rest_after = stim_id[rest_from]                  # qué instrucción precedió a cada reposo
psd_rest_hand, psd_rest_tongue = psd_rest[rest_after == 12], psd_rest[rest_after == 11]

# Lo mismo para el bloque de imaginación
seg_move_im, seg_rest_im, rest_from_im = trial_segments(t_on_im, t_off_im, V_imag_car.shape[0])
_, psd_move_im = segment_psds(V_imag_car, seg_move_im, srate)
_, psd_rest_im = segment_psds(V_imag_car, seg_rest_im, srate)
psd_hand_im, psd_tongue_im = psd_move_im[stim_im == 12], psd_move_im[stim_im == 11]
rest_after_im = stim_im[rest_from_im]
psd_rest_hand_im, psd_rest_tongue_im = psd_rest_im[rest_after_im == 12], psd_rest_im[rest_after_im == 11]

print("psd_hand:", psd_hand.shape, "→ (ensayos, canales, frecuencias)")

### 🧠 r² con signo: ¿qué tanto separa la condición a los datos?

Para cada canal y frecuencia comparamos la potencia de los ensayos de movimiento contra los de
reposo. El **r² con signo** es la fracción de la varianza total explicada por la condición
(movimiento o reposo), con signo positivo si la potencia **sube** con el movimiento y negativo si
**baja**. Es la métrica de activación de Miller et al. (2010):

$$A = \operatorname{sign}(\bar m - \bar r)\,\frac{(\bar m - \bar r)^2}{\sigma^2_{m\cup r}}\,\frac{N_m N_r}{N^2_{m\cup r}}$$

Con esta métrica escogemos automáticamente dos canales. No basta con que un canal "se active" con la
mano: queremos el más **específico**, es decir, el que más separa mano de lengua (r² mano vs lengua),
entre los canales que sí se activan respecto al reposo (r² > 0.1).

In [ ]:
def signed_r2(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """r² con signo entre dos distribuciones (Miller et al., 2010).

    Parameters
    ----------
    a, b : np.ndarray
        Muestras de cada condición; el eje 0 son los ensayos.

    Returns
    -------
    np.ndarray
        r² con signo, con la forma de ``a.shape[1:]``. Positivo si a > b.
    """
    n_a, n_b = len(a), len(b)
    diff = a.mean(axis=0) - b.mean(axis=0)
    total_var = np.concatenate([a, b], axis=0).var(axis=0)
    return np.sign(diff) * diff**2 / total_var * n_a * n_b / (n_a + n_b) ** 2


LFB, HFB = (8, 32), (76, 100)
r2_hfb_hand = signed_r2(np.log(band_power(psd_hand, freqs, HFB)), np.log(band_power(psd_rest_hand, freqs, HFB)))
r2_hfb_tongue = signed_r2(np.log(band_power(psd_tongue, freqs, HFB)), np.log(band_power(psd_rest_tongue, freqs, HFB)))
r2_hand_vs_tongue = signed_r2(np.log(band_power(psd_hand, freqs, HFB)), np.log(band_power(psd_tongue, freqs, HFB)))

active_h, active_t = r2_hfb_hand > 0.1, r2_hfb_tongue > 0.1
CH_HAND = int(np.argmax(np.where(active_h, r2_hand_vs_tongue, -np.inf))) if active_h.any() else int(np.argmax(r2_hfb_hand))
CH_TONGUE = int(np.argmin(np.where(active_t, r2_hand_vs_tongue, np.inf))) if active_t.any() else int(np.argmax(r2_hfb_tongue))
for name, chn, r2_rest in [("mano", CH_HAND, r2_hfb_hand), ("lengua", CH_TONGUE, r2_hfb_tongue)]:
    print(f"Canal de {name}: {chn} ({areas[chn]}) · r² vs reposo = {r2_rest[chn]:.2f} · "
          f"r² mano vs lengua = {r2_hand_vs_tongue[chn]:+.2f}")

In [ ]:
CHANNEL_PSD = -1  #@param {type:"integer"}
ch = CH_HAND if CHANNEL_PSD < 0 else CHANNEL_PSD   # -1 = usar el mejor canal de mano

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (ph, pr, ttl) in zip(axes, [(psd_hand, psd_rest_hand, "Movimiento"),
                                     (psd_hand_im, psd_rest_hand_im, "Imaginación")]):
    # Ocultamos los bins del notch (NaN) para que no dominen la escala vertical
    ax.semilogy(freqs, np.where(notch, np.nan, ph[:, ch].mean(0)), color=C_HAND, label="Mano")
    ax.semilogy(freqs, np.where(notch, np.nan, pr[:, ch].mean(0)), color=C_REST, label="Reposo")
    for band, col in [(LFB, C_LFB), (HFB, C_HFB)]:
        ax.axvspan(*band, color=col, alpha=0.2)
    ax.set(title=f"{ttl} · canal {ch}", xlabel="Frecuencia (Hz)", ylabel="Potencia", xlim=(0, 150))
    ax.legend()

# ✍️ TODO 4: calcula el r² con signo, frecuencia por frecuencia, entre el log de la PSD
#            de mano y el de reposo para el canal ch.  Pista: psd_hand[:, ch, :]
r2_spectrum = ...
plt.show()

In [ ]:
#@title ✔️ Verificar TODO 4 y graficar el espectro de r²
verify("r2_spectrum", signed_r2(np.log(psd_hand[:, ch, :]), np.log(psd_rest_hand[:, ch, :])),
       "Pista: signed_r2(np.log(A), np.log(B)) con A y B de forma (ensayos, frecuencias).")
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(freqs, np.where(notch, np.nan, r2_spectrum), color="k")
ax.axhline(0, color="gray", lw=0.8)
for band, col in [(LFB, C_LFB), (HFB, C_HFB)]:
    ax.axvspan(*band, color=col, alpha=0.2)
ax.set(title=f"r² con signo, mano vs reposo · canal {ch}", xlabel="Frecuencia (Hz)", xlim=(0, 150))
plt.show()

💡 ¿En qué frecuencias el r² es negativo y en cuáles positivo? ¿El cambio en alta frecuencia se
parece a un **pico** o a un desplazamiento **de todo el espectro**? Compara con la imaginación:
¿mismo patrón, menor magnitud?

### 🧠 Modelo 1: separar el espectro en "fondo 1/f" + oscilaciones

El espectro de la corteza tiene dos componentes que conviene separar (Miller et al., 2009;
Donoghue et al., 2020):

- Un **componente aperiódico** que cae como una ley de potencias, $P(f) \propto 1/f^{\chi}$.
  En escala log-log es casi una recta (con una posible "rodilla").
- **Picos oscilatorios**, como el ritmo *beta* (~15–30 Hz), que sobresalen de ese fondo.

El modelo de *specparam* es:

$$\log_{10} P(f) = b - \log_{10}\left(k + f^{\chi}\right) + \sum_i G_i(f)$$

donde $b$ es el *offset* (nivel general), $k$ la rodilla, $\chi$ el exponente y $G_i$ gaussianas
para los picos. En el rango de 3 a 100 Hz estos datos se ajustan bien sin rodilla ($k = 0$, modo
`fixed`): en log-log, una recta más los picos.

<img src="https://raw.githubusercontent.com/EmmanuelIsaiasGuizarBayardo/taller-ecog/main/img/miller2012_fig2.jpg" width="620">

*Fuente: Miller et al. (2012), PLoS Comput. Biol., Fig. 2 (CC BY). En B, el espectro reconstruido
sin las componentes rítmicas: el movimiento eleva todo el espectro. En C, solo las componentes
rítmicas: el movimiento reduce el pico beta.*

⚠️ **Evidencia y debate.** Que el *offset* de banda ancha siga la tasa de disparo local tiene buen
respaldo (Manning et al., 2009; Miller et al., 2009). Interpretar el **exponente** como marcador
del balance excitación/inhibición es una hipótesis con apoyo parcial y todavía debatida
(Gao et al., 2017).

In [ ]:
from specparam import SpectralModel

APERIODIC_MODE = "fixed"   #@param ["fixed", "knee"]
FREQ_RANGE = [3, 100]


def interpolate_notches(f: np.ndarray, p: np.ndarray,
                        centers: tuple[float, ...] = (60, 120), half_width: float = 4) -> np.ndarray:
    """Interpola linealmente (en log-log) los huecos dejados por el filtro notch."""
    p = p.copy()
    for c in centers:
        gap = (f >= c - half_width) & (f <= c + half_width)
        if gap.any():
            edges = ~gap & (np.abs(f - c) <= 3 * half_width)
            p[gap] = 10 ** np.interp(np.log10(f[gap]), np.log10(f[edges]), np.log10(p[edges]))
    return p


spec_rest = interpolate_notches(freqs, psd_rest_hand[:, CH_HAND].mean(0))
spec_hand = interpolate_notches(freqs, psd_hand[:, CH_HAND].mean(0))
model_kwargs = dict(aperiodic_mode=APERIODIC_MODE, peak_width_limits=(2, 12),
                    max_n_peaks=4, min_peak_height=0.05, verbose=False)

fm_rest = SpectralModel(**model_kwargs)
fm_rest.fit(freqs, spec_rest, FREQ_RANGE)

# ✍️ TODO 5: crea y ajusta el modelo para el espectro de movimiento de mano (spec_hand),
#            igual que se hizo arriba para el reposo.
fm_hand = ...
# fm_hand.fit(...)   <- descomenta esta línea y complétala

In [ ]:
#@title ✔️ Verificar TODO 5 y comparar ambos modelos
if not isinstance(globals().get("fm_hand"), SpectralModel) or not fm_hand.results.has_model:
    print("❌ fm_hand: todavía no hay un modelo ajustado. Pista: copia las dos líneas de fm_rest.")
    print("   Usamos el modelo de referencia para que puedas continuar.")
    fm_hand = SpectralModel(**model_kwargs)
    fm_hand.fit(freqs, spec_hand, FREQ_RANGE)
else:
    print("✅ fm_hand: ¡modelo ajustado!")
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, fm, ttl in zip(axes, [fm_rest, fm_hand], ["Reposo", "Movimiento de mano"]):
    fm.plot(ax=ax, plt_log=True)
    ax.set_title(f"{ttl} · canal {CH_HAND}")
plt.show()


def summarize(fm: "SpectralModel") -> dict:
    """Parámetros clave: offset, exponente, potencia aperiódica a 90 Hz y pico beta (13–30 Hz)."""
    ap = fm.get_params("aperiodic")
    knee = ap[1] if len(ap) == 3 else 0.0
    peaks = np.atleast_2d(fm.get_params("peak"))
    beta = [pk for pk in peaks if not np.isnan(pk[0]) and 13 <= pk[0] <= 30]
    return {"offset": ap[0], "exponente": ap[-1],
            "aperiódico a 90 Hz": ap[0] - np.log10(knee + 90 ** ap[-1]),
            "pico beta (altura)": max((pk[1] for pk in beta), default=0.0),
            "R² del ajuste": fm.get_metrics("gof_rsquared")}


s_rest, s_hand = summarize(fm_rest), summarize(fm_hand)
print(f"{'parámetro (log10)':<22}{'reposo':>10}{'mano':>10}")
for key in s_rest:
    print(f"{key:<22}{s_rest[key]:>10.3f}{s_hand[key]:>10.3f}")

💡 ¿Qué cambió con el movimiento: el *offset* (banda ancha), el exponente o el pico beta?
¿Cómo encaja esto con la Fig. 2 de Miller et al. (2012)?

## ☕ Descanso (10 min)

## 3. La envolvente de alta frecuencia en el tiempo

El espectro promedia ventanas de varios segundos. Para una BCI necesitamos seguir la actividad
**en tiempo casi real**. La receta clásica tiene tres pasos:

1. **Filtro pasa-banda** (76–100 Hz): nos quedamos solo con esa banda.
2. **Transformada de Hilbert:** convierte la señal filtrada en una señal compleja (analítica)
   cuyo módulo es la **amplitud instantánea**.
3. **Potencia = |señal analítica|²**, y opcionalmente un suavizado.

<img src="https://raw.githubusercontent.com/EmmanuelIsaiasGuizarBayardo/taller-ecog/main/img/miller2012_fig1.jpg" width="560">

*Fuente: Miller et al. (2012), PLoS Comput. Biol., Fig. 1 (CC BY). Panel D: posición de los
dedos, cambio de banda ancha, ritmo motor y voltaje crudo, todos en el mismo eje de tiempo.*

In [ ]:
BAND_LO = 76    #@param {type:"slider", min:50, max:150, step:1}
BAND_HI = 100   #@param {type:"slider", min:60, max:200, step:1}

sos = signal.butter(4, [BAND_LO, BAND_HI], btype="bandpass", fs=srate, output="sos")
x_bp = signal.sosfiltfilt(sos, V_car[:, CH_HAND])

# ✍️ TODO 6: calcula la señal analítica con signal.hilbert y luego su potencia instantánea.
analytic = ...
power = ...

In [ ]:
#@title ✔️ Verificar TODO 6 y graficar
verify("power", np.abs(signal.hilbert(x_bp)) ** 2, "Pista: np.abs(analytic) ** 2")
seg = slice(START_S * srate, (START_S + 20) * srate)
tt = np.arange(seg.start, seg.stop) / srate
fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
axes[0].plot(tt, V_car[seg, CH_HAND], "k", lw=0.4); axes[0].set_ylabel("Cruda (µV)")
axes[1].plot(tt, x_bp[seg], color=C_HFB, lw=0.4); axes[1].set_ylabel(f"{BAND_LO}–{BAND_HI} Hz")
_sos5 = signal.butter(2, 5, fs=srate, output="sos")
axes[2].plot(tt, power[seg], color="purple", lw=0.4, alpha=0.35)
axes[2].plot(tt, signal.sosfiltfilt(_sos5, power)[seg], color="purple", lw=1.5, label="suavizada (5 Hz)")
axes[2].set_ylabel("Potencia (µV²)"); axes[2].legend(loc="upper right")
axes[2].set_ylim(0, np.percentile(power[seg], 99.5))
for ax in axes:
    for a, b, s in zip(t_on, t_off, stim_id):
        if b > seg.start and a < seg.stop:
            ax.axvspan(max(a, seg.start) / srate, min(b, seg.stop) / srate, alpha=0.15,
                       color=C_HAND if s == 12 else C_TONGUE)
axes[2].set_xlabel("Tiempo (s)")
axes[0].set_title(f"Canal {CH_HAND} · rojo: mano, azul: lengua")
plt.show()

Ahora aplicamos lo mismo a **todos los canales** (con la función `band_envelope`, que hace
exactamente tus pasos) y cortamos **épocas** de −1 a +3 s alrededor de cada instrucción.

🧠 **Normalización respecto a la línea base.** Expresamos la potencia como **cambio porcentual**
respecto al periodo previo a la instrucción (−1 a 0 s), promediado sobre todos los ensayos de cada
canal. Así comparamos canales con niveles de potencia distintos. (Usar el promedio de todos los
ensayos, y no la base de cada ensayo por separado, hace la referencia mucho menos ruidosa.)

In [ ]:
SMOOTH_HZ = 10   #@param {type:"slider", min:2, max:30, step:1}
env_hfb = band_envelope(V_car, srate, (BAND_LO, BAND_HI), smooth_hz=SMOOTH_HZ)
env_lfb = band_envelope(V_car, srate, LFB, smooth_hz=min(SMOOTH_HZ, 3))   # la LFB cambia más lento

ep_hfb, t_ep, keep = epoch(env_hfb, t_on, -1000, 3000, srate)
ep_lfb, _, _ = epoch(env_lfb, t_on, -1000, 3000, srate)
stim_ep = stim_id[keep]                        # tipo de instrucción de cada época

# ✍️ TODO 7: calcula la potencia promedio del periodo base (t_ep < 0) de cada canal, promediando
#            sobre ensayos (eje 0) y tiempo (eje 1); luego expresa ep_hfb como cambio porcentual.
#            Pista: ep_hfb[:, t_ep < 0, :].mean(axis=(0, 1))  ->  forma (n_canales,)
base_hfb = ...
ep_hfb_pct = ...

In [ ]:
#@title ✔️ Verificar TODO 7 y graficar la doble disociación
_base = ep_hfb[:, t_ep < 0, :].mean(axis=(0, 1))
verify("ep_hfb_pct", 100 * (ep_hfb / _base - 1), "Pista: 100 * (ep_hfb / base_hfb - 1)")
_bl = ep_lfb[:, t_ep < 0, :].mean(axis=(0, 1))
ep_lfb_pct = 100 * (ep_lfb / _bl - 1)
hand_ep = stim_ep == 12


def mean_sem(arr: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    return arr.mean(0), arr.std(0) / np.sqrt(len(arr))


fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)
for col, (chn, name) in enumerate([(CH_HAND, "canal de mano"), (CH_TONGUE, "canal de lengua")]):
    for row, (ep, band) in enumerate([(ep_hfb_pct, "HFB"), (ep_lfb_pct, "LFB")]):
        ax = axes[row, col]
        for mask, colr, lab in [(hand_ep, C_HAND, "mano"), (~hand_ep, C_TONGUE, "lengua")]:
            m, s = mean_sem(ep[mask, :, chn])
            ax.plot(t_ep, m, color=colr, label=lab)
            ax.fill_between(t_ep, m - s, m + s, color=colr, alpha=0.25)
        ax.axvline(0, color="k", lw=0.8); ax.axhline(0, color="gray", lw=0.8)
        ax.set(title=f"{band} · {name} ({chn})", ylabel="% cambio")
        ax.legend()
axes[1, 0].set_xlabel("Tiempo desde la instrucción (ms)")
axes[1, 1].set_xlabel("Tiempo desde la instrucción (ms)")
plt.show()

💡 Esto es una **doble disociación**: el canal de mano responde a la mano y no a la lengua, y
viceversa. ¿Pasa lo mismo en LFB? ¿Qué te dice eso sobre la **especificidad espacial** de cada
banda?

In [ ]:
#@title 🔍 Ensayo por ensayo (raster)
order = np.argsort(stim_ep, kind="stable")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, chn, name in zip(axes, [CH_HAND, CH_TONGUE], ["mano", "lengua"]):
    im = ax.imshow(ep_hfb_pct[order, :, chn], aspect="auto", cmap="magma", vmin=-50, vmax=300,
                   extent=[t_ep[0], t_ep[-1], len(order), 0])
    ax.axhline(np.sum(stim_ep == 11), color="w", ls="--")
    ax.set(title=f"HFB, canal de {name} ({chn}) · arriba: lengua, abajo: mano",
           xlabel="Tiempo (ms)", ylabel="Ensayo")
    plt.colorbar(im, ax=ax, label="% cambio")
plt.show()

### 💡 ¿Cerebro o músculo?

Un canal puede "responder" a una tarea por razones que no son corticales. Abrir la boca y mover la
lengua activa músculos de la mandíbula y la sien, y su actividad eléctrica (EMG) es de **banda muy
ancha**. ECoG es mucho menos vulnerable al EMG que el EEG (Schalk y Leuthardt, 2011), pero no es
inmune, sobre todo en electrodos del borde de la rejilla cerca del músculo temporal. Dos pistas:

- **Actividad cortical motora:** la LFB baja y la HFB sube, con un efecto que se atenúa en las
  frecuencias más altas.
- **Posible EMG:** aumento plano de ~40 Hz hasta 200 Hz, **sin** caída en LFB, en un electrodo
  lejos de la corteza motora.

Sin un registro de EMG simultáneo no se puede confirmar; es una hipótesis informada.

In [ ]:
#@title 🔍 Perfil espectral de los dos canales elegidos (0–200 Hz)
fig, ax = plt.subplots(figsize=(10, 3.8))
for chn, pm_, pr_, col, lab in [(CH_HAND, psd_hand, psd_rest_hand, C_HAND, "mano"),
                                (CH_TONGUE, psd_tongue, psd_rest_tongue, C_TONGUE, "lengua")]:
    r2_full = signed_r2(np.log(pm_[:, chn, :]), np.log(pr_[:, chn, :]))
    ax.plot(freqs, np.where(notch, np.nan, r2_full), color=col,
            label=f"canal {chn} ({areas[chn]}), {lab} vs reposo")
    print(f"Canal {chn}: {areas[chn]} · coordenada z (Talairach) = {dat_real['locs'][good][chn][2]:.0f} mm")
ax.axhline(0, color="gray", lw=0.8)
for band, col in [(LFB, C_LFB), (HFB, C_HFB)]:
    ax.axvspan(*band, color=col, alpha=0.15)
ax.set(xlabel="Frecuencia (Hz)", ylabel="r² con signo", xlim=(0, 200))
ax.legend(fontsize=9); plt.show()

## 4. Mapeo funcional

### 🎯 Objetivo de réplica: Miller et al. (2010), Fig. 1C–D

En la Fig. 1C (arriba en este notebook) los mapas de **HFB** son focales y distintos para mano y
lengua; en la Fig. 1D los de **LFB** son amplios y se traslapan. Vamos a calcular el r² con signo
de cada canal para las cuatro combinaciones (mano/lengua × real/imaginado) y dibujarlo sobre el
cerebro.

In [ ]:
def activation_map(psd_task: np.ndarray, psd_base: np.ndarray, band: tuple) -> np.ndarray:
    """r² con signo por canal entre el log de la potencia en banda de dos condiciones."""
    return signed_r2(np.log(band_power(psd_task, freqs, band)), np.log(band_power(psd_base, freqs, band)))


conds = {
    ("mano", "real"): (psd_hand, psd_rest_hand), ("lengua", "real"): (psd_tongue, psd_rest_tongue),
    ("mano", "imag"): (psd_hand_im, psd_rest_hand_im), ("lengua", "imag"): (psd_tongue_im, psd_rest_tongue_im),
}
maps = {band_name: {k: activation_map(*v, band) for k, v in conds.items()}
        for band_name, band in [("HFB", HFB), ("LFB", LFB)]}

BAND_TO_PLOT = "HFB"  #@param ["HFB", "LFB"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for i, kind in enumerate(["real", "imag"]):
    for j, eff in enumerate(["mano", "lengua"]):
        vals = maps[BAND_TO_PLOT][(eff, kind)]
        plot_brain(vals, locs_mni, ax=axes[i, j],
                   title=f"{BAND_TO_PLOT} · {eff} · {kind}  (max |r²| = {np.abs(vals).max():.2f})")
plt.show()

### 🧠 ¿Cuánto se parecen dos mapas? Métrica de traslape con permutaciones

Miller et al. (2010) cuantifican el parecido de dos mapas con el **producto punto** de sus
activaciones. Para saber si ese valor es "grande", lo comparan con una distribución nula: barajan
al azar la asignación de canales de uno de los mapas miles de veces. El resultado se expresa como
**z-score** respecto a esa distribución nula.

In [ ]:
map_a = maps["HFB"][("mano", "real")]
map_b = maps["HFB"][("mano", "imag")]
# ✍️ TODO 8: el traslape entre dos mapas es su producto punto: multiplica canal por canal y suma.
#            Pista: np.dot(map_a, map_b)
dot_example = ...

In [ ]:
#@title ✔️ Verificar TODO 8
verify("dot_example", np.dot(map_a, map_b), "Pista: np.dot(map_a, map_b)")
print(f"Producto punto mano real · mano imaginada (HFB): {dot_example:.4f}")
print("¿Es grande o pequeño? Sin una distribución nula no podemos saberlo. Eso hace la celda siguiente.")

In [ ]:
def overlap_metric(a: np.ndarray, b: np.ndarray, n_perm: int = 10_000,
                   seed: int = 0) -> tuple[float, float]:
    """Traslape entre mapas con prueba de permutación (Miller et al., 2010).

    Returns
    -------
    z : float
        Producto punto observado en unidades z de la distribución nula.
    p : float
        Fracción de permutaciones con producto punto mayor o igual al observado.
    """
    rng = np.random.default_rng(seed)
    observed = np.dot(a, b)
    perms = rng.random((n_perm, len(b))).argsort(axis=1)   # n_perm barajadas de canales
    null = (b[perms] * a).sum(axis=1)
    return (observed - null.mean()) / null.std(), float(np.mean(null >= observed))


pairs = [("mano real vs lengua real", ("mano", "real"), ("lengua", "real")),
         ("mano real vs mano imag", ("mano", "real"), ("mano", "imag")),
         ("lengua real vs lengua imag", ("lengua", "real"), ("lengua", "imag"))]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, band_name in zip(axes, ["HFB", "LFB"]):
    zs = []
    for label, k1, k2 in pairs:
        z, p = overlap_metric(maps[band_name][k1], maps[band_name][k2])
        zs.append(z)
        print(f"{band_name} | {label:<28} z = {z:5.2f}  p = {p:.4f}")
    ax.bar(range(3), zs, color=["#f2e85c", "#8fe3f0", "#e07be0"])
    ax.axhline(2.33, color="gray", ls="--", lw=0.8)   # ~p < 0.01 unilateral
    ax.set(xticks=range(3), xticklabels=["mano–lengua\n(real)", "mano\nreal–imag", "lengua\nreal–imag"],
           ylabel="Traslape (z)", title=band_name)
plt.show()

### 🎯 Objetivo de réplica: Miller et al. (2010), Fig. 2A

<img src="https://raw.githubusercontent.com/EmmanuelIsaiasGuizarBayardo/taller-ecog/main/img/miller2010_fig2.jpg" width="520">

*Fuente: Miller et al. (2010), PNAS, Fig. 2. Uso educativo no comercial.*

En los electrodos que responden al movimiento, ¿qué fracción del cambio aparece durante la
imaginación? En el artículo (8 sujetos): **0.26 en HFB** y **0.49 en LFB**.

In [ ]:
#@title 📏 Cociente imaginación : movimiento
ratios = {"HFB": [], "LFB": []}
for eff, (pm, pr, pmi, pri) in {"mano": (psd_hand, psd_rest_hand, psd_hand_im, psd_rest_hand_im),
                                "lengua": (psd_tongue, psd_rest_tongue, psd_tongue_im, psd_rest_tongue_im)}.items():
    # Electrodos con cambio significativo en AMBAS bandas durante el movimiento (Bonferroni)
    sig = np.ones(n_channels, bool)
    for band in (HFB, LFB):
        _, pval = stats.ttest_ind(band_power(pm, freqs, band), band_power(pr, freqs, band), axis=0)
        sig &= pval < 0.05 / n_channels
    for name, band in [("HFB", HFB), ("LFB", LFB)]:
        shift_real = band_power(pm, freqs, band).mean(0) - band_power(pr, freqs, band).mean(0)
        shift_imag = band_power(pmi, freqs, band).mean(0) - band_power(pri, freqs, band).mean(0)
        ratios[name] += list((shift_imag / shift_real)[sig])

fig, ax = plt.subplots(figsize=(5, 4.5))
for i, (name, col, ref) in enumerate([("LFB", C_LFB, 0.49), ("HFB", C_HFB, 0.26)]):
    r = np.array(ratios[name])
    gm = np.exp(np.mean(np.log(r[r > 0]))) if np.any(r > 0) else np.nan
    jitter = np.random.default_rng(i).uniform(-0.08, 0.08, len(r))
    ax.scatter(np.full(len(r), i) + jitter, r, color=col, s=18)
    ax.hlines(gm, i - 0.3, i + 0.3, color=col, lw=3, label=f"{name}: {gm:.2f} (artículo {ref})")
    print(f"{name}: media geométrica = {gm:.2f} con {len(r)} electrodos (artículo: {ref})")
    ax.hlines(ref, i - 0.3, i + 0.3, color="k", ls=":", lw=1.5)
ax.set(xticks=[0, 1], xticklabels=["LFB", "HFB"], ylabel="Imaginación : movimiento",
       ylim=(-0.5, 1.8), title=f"Sujeto {SUBJECT} · {len(ratios['HFB'])} electrodos")
ax.legend(fontsize=9); plt.show()

💡 Con un solo sujeto y pocos electrodos, ¿qué tan confiable es este número? ¿Qué harías para
compararlo de forma justa con el del artículo?

## 5. Decodificación: del mapa a la BCI

### 🧠 Modelo 2: un clasificador

Ahora la pregunta cambia: con la actividad de **un solo ensayo**, ¿podemos adivinar si la persona
movió la mano o la lengua? Para cada ensayo construimos un vector de **características**
(*features*): el log de la potencia HFB y LFB de cada canal. Luego entrenamos una **regresión
logística** y medimos su exactitud con **validación cruzada**: entrenamos con una parte de los
ensayos y evaluamos con otra que el modelo nunca vio.

⚠️ Tenemos ~60 ensayos y ~2 × (número de canales) *features*: más variables que ejemplos. La
regularización de la regresión logística evita que el modelo memorice.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, permutation_test_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def features(psd: np.ndarray, which: str = "both") -> np.ndarray:
    """Matriz de features (ensayos × features): log de potencia en HFB y/o LFB por canal."""
    parts = {"HFB": [HFB], "LFB": [LFB], "both": [HFB, LFB]}[which]
    return np.hstack([np.log(band_power(psd, freqs, b)) for b in parts])


y = is_hand.astype(int)                       # 1 = mano, 0 = lengua
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=2000))

results = {}
for which in ["LFB", "HFB", "both"]:
    X = features(psd_move, which)
    score, _, p = permutation_test_score(clf, X, y, cv=cv, n_permutations=200, random_state=0)
    results[which] = score
    print(f"{which:>5}: exactitud = {score:.2f}  (p de permutación = {p:.3f})")

plt.bar(results.keys(), results.values(), color=[C_LFB, C_HFB, "gray"])
plt.axhline(0.5, color="k", ls="--", label="azar teórico")
plt.ylim(0, 1); plt.ylabel("Exactitud (CV 5-fold)"); plt.legend(); plt.show()

### ⚠️ El error más común: *data leakage*

Supón que primero escoges los 5 "mejores" canales usando **todos** los ensayos, y después haces
validación cruzada solo con esos canales. Parece razonable, pero el conjunto de prueba ya influyó
en la selección. Para demostrarlo, vamos a usar **etiquetas al azar**: no hay nada que aprender,
así que la exactitud honesta debería rondar el 50%.

In [ ]:
K = 5   #@param {type:"slider", min:1, max:20, step:1}
X_all = features(psd_move, "both")
rng_leak = np.random.default_rng(1)

# ✍️ TODO 9: construye el pipeline CORRECTO, con la selección de features DENTRO
#            de la validación cruzada: StandardScaler -> SelectKBest(f_classif, k=K) -> LogisticRegression
pipe_ok = ...

In [ ]:
#@title ✔️ Verificar TODO 9 y correr el experimento
from sklearn.pipeline import Pipeline
_ok = isinstance(globals().get("pipe_ok"), Pipeline) and any(
    isinstance(step, SelectKBest) for _, step in pipe_ok.steps)
if _ok:
    print("✅ pipe_ok: ¡el pipeline incluye la selección de features!")
else:
    print("❌ pipe_ok: falta un Pipeline con SelectKBest adentro. Usamos el de referencia.")
    pipe_ok = make_pipeline(StandardScaler(), SelectKBest(f_classif, k=K),
                            LogisticRegression(C=0.1, max_iter=2000))

wrong, right = [], []
for rep in range(20):
    y_fake = rng_leak.permutation(y)
    # ❌ Incorrecto: se seleccionan features con TODOS los ensayos, antes de la CV
    X_sel = SelectKBest(f_classif, k=K).fit_transform(X_all, y_fake)
    wrong.append(cross_val_score(clf, X_sel, y_fake, cv=cv).mean())
    # ✅ Correcto: la selección ocurre dentro de cada fold
    right.append(cross_val_score(pipe_ok, X_all, y_fake, cv=cv).mean())

fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot([wrong, right])
ax.set_xticks([1, 2], ["selección fuera\nde la CV (incorrecto)", "selección dentro\nde la CV (correcto)"])
ax.axhline(0.5, color="k", ls="--"); ax.set_ylabel("Exactitud con etiquetas AL AZAR")
ax.set_title("Data leakage"); plt.show()
print(f"Fuera de la CV: {np.mean(wrong):.2f} | Dentro de la CV: {np.mean(right):.2f}")

💡 Con etiquetas aleatorias, el método incorrecto "encuentra" una exactitud muy por encima del
azar. ¿Cuántos artículos crees que reportan exactitudes infladas por este error?

### Transferencia: entrenar con movimiento real, probar con imaginación

En una BCI para una persona con parálisis no hay movimiento real para entrenar. ¿Un decodificador
entrenado con movimiento real reconoce la **imaginación**? Como la imaginación produce cambios más
pequeños (~25%), estandarizamos las *features* de cada bloque por separado: es una forma muy
simple de **adaptación de dominio**.

In [ ]:
X_real, X_imag = features(psd_move), features(psd_move_im)
y_imag = (stim_im[: len(X_imag)] == 12).astype(int)
model = LogisticRegression(C=0.1, max_iter=2000)
model.fit(StandardScaler().fit_transform(X_real), y)
acc_raw = model.score(StandardScaler().fit(X_real).transform(X_imag), y_imag)
acc_adapt = model.score(StandardScaler().fit_transform(X_imag), y_imag)
print(f"Real → imaginación, escalado con el bloque real:   {acc_raw:.2f}")
print(f"Real → imaginación, escalado con su propio bloque: {acc_adapt:.2f}")

### 🎯 Una BCI de una dimensión

<img src="https://raw.githubusercontent.com/EmmanuelIsaiasGuizarBayardo/taller-ecog/main/img/miller2010_fig3.jpg" width="430">

*Fuente: Miller et al. (2010), PNAS, Fig. 3. Uso educativo no comercial.*

En el experimento de retroalimentación, la velocidad del cursor era proporcional a la potencia HFB
de **un solo electrodo** respecto a su media: $\dot y(t) = g\,(P(t) - P_0)$. Vamos a simularlo
*offline* con los ensayos de imaginación: si la persona imagina la mano, el cursor debería subir.

⚠️ **¿Qué electrodo usamos?** En el artículo, el electrodo y la banda se eligieron con una sesión
de tamizaje previa. Aquí lo imitamos: elegimos el canal con la **primera mitad** de los ensayos de
imaginación y medimos la exactitud solo con la **segunda mitad**. Si eligiéramos y evaluáramos con
los mismos ensayos, cometeríamos el mismo *leakage* de antes.

In [ ]:
GAIN = 1.0   #@param {type:"slider", min:0.1, max:5, step:0.1}
# 1) Tamizaje: con la PRIMERA mitad de los ensayos elegimos el canal más discriminativo
first_half = np.arange(len(stim_im)) < len(stim_im) // 2
hfb_im = np.log(band_power(psd_move_im, freqs, HFB))
r2_screen = signed_r2(hfb_im[first_half & (stim_im == 12)], hfb_im[first_half & (stim_im == 11)])
CH_BCI = int(np.argmax(np.abs(r2_screen)))
direction = np.sign(r2_screen[CH_BCI])           # +1 si el canal prefiere mano, -1 si prefiere lengua

# 2) Cursor: velocidad proporcional a la potencia HFB respecto a P0 (la media del tamizaje)
env_im = band_envelope(V_imag_car[:, [CH_BCI]], srate, HFB, smooth_hz=SMOOTH_HZ)[:, 0]
ep_im, t_ep_im, keep_im = epoch(env_im[:, None], t_on_im, 0, 3000, srate)
P = np.log(ep_im[:, :, 0])
train = first_half[keep_im]
P0 = P[train].mean()
velocity = direction * GAIN * (P - P0)           # orientado para que "mano" suba
cursor = np.cumsum(velocity, axis=1) / srate      # integrar la velocidad en el tiempo

# 3) Evaluación SOLO con la segunda mitad
hand_im = stim_im[keep_im] == 12
fig, ax = plt.subplots(figsize=(10, 4))
for traj, h in zip(cursor[~train], hand_im[~train]):
    ax.plot(t_ep_im / 1000, traj, color=C_HAND if h else C_TONGUE, alpha=0.5, lw=1)
ax.axhline(0, color="k", lw=0.8)
acc_cursor = np.mean((cursor[~train, -1] > 0) == hand_im[~train])
print(f"Canal de tamizaje: {CH_BCI} ({areas[CH_BCI]}), r² = {r2_screen[CH_BCI]:+.2f} · exactitud en la 2.ª mitad: {acc_cursor:.2f}")
ax.set(title=f"Cursor simulado (2.ª mitad) · canal {CH_BCI} ({areas[CH_BCI]}) · exactitud = {acc_cursor:.2f}",
       xlabel="Tiempo (s)", ylabel="Posición del cursor (rojo: mano, azul: lengua)")
plt.show()

💡 Esta simulación es de **lazo abierto**: el paciente no veía el cursor. En el experimento real
(lazo cerrado), la actividad durante la imaginación **aumentó** con la práctica y en varios casos
superó a la del movimiento real (Miller et al., 2010, Figs. 4 y 5). ¿Por qué crees que la
retroalimentación cambia la señal?

<img src="https://raw.githubusercontent.com/EmmanuelIsaiasGuizarBayardo/taller-ecog/main/img/miller2010_fig5.jpg" width="380">

## 6. Extra: movimientos individuales de los dedos

Otro grupo de pacientes movió **un dedo a la vez** mientras un guante medía la flexión de los
cinco dedos (Miller et al., 2009). Si la banda ancha refleja actividad local, cada dedo debería
tener "su" electrodo.

<img src="https://raw.githubusercontent.com/EmmanuelIsaiasGuizarBayardo/taller-ecog/main/img/miller2012_fig3.jpg" width="560">

*Fuente: Miller et al. (2012), PLoS Comput. Biol., Fig. 3 (CC BY). Somatotopía de los dedos con
banda ancha (arriba) contra ritmo beta (abajo).*

In [ ]:
#@title 💾 Cargar *fingerflex* y calcular la envolvente de banda ancha
alldat_ff = load_npz_dataset("fingerflex")
FF_SUBJECT = 0   #@param {type:"slider", min:0, max:2, step:1}
records_ff = list(np.ravel(alldat_ff))                    # un registro por sujeto, sin importar la forma del arreglo
dat_ff = records_ff[min(FF_SUBJECT, len(records_ff) - 1)]
srate_ff = int(np.squeeze(dat_ff["srate"]))
V_ff = dat_ff["V"].astype("float32")
V_ff -= V_ff.mean(axis=1, keepdims=True)                 # CAR
dg = dat_ff["dg"].astype("float32")                       # flexión de los 5 dedos
finger_names = ["pulgar", "índice", "medio", "anular", "meñique"]

# Envolvente 70–110 Hz suavizada a 3 Hz; luego bajamos a 100 Hz para que todo sea rápido
env_ff = band_envelope(V_ff, srate_ff, (70, 110), smooth_hz=3)
DS = 10
env_ds = np.log(env_ff[::DS])
dg_ds = dg[::DS]
fs_ds = srate_ff // DS
print("Envolvente:", env_ds.shape, "| dedos:", dg_ds.shape)

In [ ]:
corr = np.array([[np.corrcoef(env_ds[:, c], dg_ds[:, f])[0, 1] for f in range(5)]
                 for c in range(env_ds.shape[1])])
best_ch = corr.argmax(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(15, 4), gridspec_kw={"width_ratios": [2, 1]})
im = axes[0].imshow(corr.T, aspect="auto", cmap="RdBu_r", vmin=-0.5, vmax=0.5)
axes[0].set(yticks=range(5), yticklabels=finger_names, xlabel="Canal",
            title="Correlación banda ancha ↔ flexión")
plt.colorbar(im, ax=axes[0])
axes[1].bar(finger_names, corr[best_ch, range(5)], color="purple")
axes[1].set(title="Mejor canal por dedo", ylabel="r")
for i, c in enumerate(best_ch):
    axes[1].text(i, 0.01, f"ch{c}", ha="center", color="w", fontsize=9)
plt.show()

In [ ]:
#@title ⏱️ ¿La corteza se adelanta al dedo? Correlación cruzada
FINGER = 0   #@param {type:"slider", min:0, max:4, step:1}
ch_f = best_ch[FINGER]
a = stats.zscore(env_ds[:, ch_f]); b = stats.zscore(dg_ds[:, FINGER])
lags = np.arange(-40, 41)                                   # ±400 ms a 100 Hz
xc = [np.mean(a[max(0, -L): len(a) - max(0, L)] * b[max(0, L): len(b) - max(0, -L)]) for L in lags]
best_lag_ms = lags[int(np.argmax(xc))] * 1000 / fs_ds

fig, axes = plt.subplots(1, 2, figsize=(15, 3.5), gridspec_kw={"width_ratios": [2.5, 1]})
# Ventana de 40 s que empieza 2 s antes de la primera instrucción de ese dedo
t0 = max(0, int(dat_ff["t_on"][dat_ff["stim_id"] == FINGER + 1][0] / DS) - 2 * fs_ds)
seg = slice(t0, t0 + 40 * fs_ds)
tt = np.arange(seg.start, seg.stop) / fs_ds
axes[0].plot(tt, b[seg], color="k", label=f"{finger_names[FINGER]} (guante)")
axes[0].plot(tt, a[seg] - 4, color="purple", label=f"banda ancha, canal {ch_f}")
axes[0].set(xlabel="Tiempo (s)", yticks=[]); axes[0].legend(loc="upper right")
axes[1].plot(lags * 1000 / fs_ds, xc, color="k"); axes[1].axvline(best_lag_ms, color="r")
axes[1].set(xlabel="Desfase cerebro → dedo (ms)", ylabel="Correlación",
            title=f"Pico: {best_lag_ms:.0f} ms (artículo: ~84 ms)")
print(f"{finger_names[FINGER]}: canal {ch_f}, r = {corr[ch_f, FINGER]:.2f}, desfase de máxima correlación = {best_lag_ms:.0f} ms")
plt.show()

### Decodificar la posición del dedo (regresión)

Un último modelo: predecir la flexión de cada dedo a partir de la banda ancha de **todos** los
canales, con varios retrasos temporales, usando regresión *ridge*.

🧠 **Por qué la división es temporal.** Las muestras vecinas en el tiempo son casi idénticas. Si
mezcláramos al azar, el modelo vería en el entrenamiento muestras pegadas a las de prueba: otra
forma de *leakage*. Por eso entrenamos con el primer 70% del registro y probamos con el 30% final.

In [ ]:
from sklearn.linear_model import Ridge

LAGS_MS = [0, 50, 100, 150, 200]                 # la corteza ANTECEDE al movimiento
shifts = [int(L * fs_ds / 1000) for L in LAGS_MS]
n_ds = env_ds.shape[0] - max(shifts)
X_ff = np.hstack([env_ds[max(shifts) - s: max(shifts) - s + n_ds] for s in shifts])
Y_ff = dg_ds[max(shifts): max(shifts) + n_ds]

# ✍️ TODO 10: índice que separa el primer 70% (entrenamiento) del 30% final (prueba).
split = ...

In [ ]:
#@title ✔️ Verificar TODO 10 y entrenar
verify("split", int(0.7 * n_ds), "Pista: int(0.7 * n_ds); sin barajar.")
scaler = StandardScaler().fit(X_ff[:split])
ridge = Ridge(alpha=10.0).fit(scaler.transform(X_ff[:split]), Y_ff[:split])
Y_hat = ridge.predict(scaler.transform(X_ff[split:]))
r_test = [np.corrcoef(Y_hat[:, f], Y_ff[split:, f])[0, 1] for f in range(5)]
print("r de prueba por dedo:", {n: round(float(r), 2) for n, r in zip(finger_names, r_test)})

y_true = Y_ff[split:, FINGER]
t0 = max(0, int(np.argmax(y_true > np.percentile(y_true, 95))) - 3 * fs_ds)   # empezar cerca de un movimiento
seg = slice(t0, t0 + 30 * fs_ds)
tt = np.arange(seg.start, seg.stop) / fs_ds
plt.figure(figsize=(14, 3))
plt.plot(tt, y_true[seg], "k", label="real")
plt.plot(tt, Y_hat[seg, FINGER], color="purple", label="predicción")
plt.xlabel("Tiempo en el conjunto de prueba (s)"); plt.ylabel("Flexión (u. a.)")
plt.title(f"{finger_names[FINGER]} · r = {r_test[FINGER]:.2f}"); plt.legend(); plt.show()

## 7. Cierre

| Lo que hicimos | Lo que encontraron los autores |
|---|---|
| PSD movimiento vs reposo | LFB baja, HFB sube (Miller et al., 2010, Fig. 1A) |
| Modelo 1/f + picos | El cambio de alta frecuencia es de banda ancha (Miller et al., 2009) |
| Envolvente HFB y doble disociación | HFB es focal y somatotópica; LFB es difusa |
| Mapas y traslape | Imaginación ≈ movimiento en el espacio, con menor magnitud |
| Clasificador y *leakage* | La validación honesta importa más que el algoritmo |
| Cursor simulado | Base de las primeras BCI con ECoG (Leuthardt et al., 2004) |
| Dedos | Representación individual de los dedos en banda ancha |

<img src="https://raw.githubusercontent.com/EmmanuelIsaiasGuizarBayardo/taller-ecog/main/img/miller2012_fig13.jpg" width="520">

*Fuente: Miller et al. (2012), PLoS Comput. Biol., Fig. 13 (CC BY). Modelo conceptual: la banda
ancha refleja actividad local y el ritmo beta modula a la población.*

**💡 Para discutir**
1. ¿Qué ventajas y qué riesgos tiene ECoG frente a EEG y frente a microelectrodos para una BCI
   crónica?
2. Estos pacientes tenían electrodos por razones clínicas, solo durante días. ¿Qué limita eso en
   lo que podemos concluir?
3. Si tuvieras que diseñar una BCI para alguien con ELA, ¿qué banda y qué paradigma elegirías, y
   por qué?

### Referencias

- Donoghue, T., Haller, M., Peterson, E. J., Varma, P., Sebastian, P., Gao, R., Noto, T., Lara, A. H.,
  Wallis, J. D., Knight, R. T., Shestyuk, A., & Voytek, B. (2020). Parameterizing neural power spectra
  into periodic and aperiodic components. *Nature Neuroscience, 23*(12), 1655–1665.
  https://doi.org/10.1038/s41593-020-00744-x
- Gao, R., Peterson, E. J., & Voytek, B. (2017). Inferring synaptic excitation/inhibition balance
  from field potentials. *NeuroImage, 158*, 70–78. https://doi.org/10.1016/j.neuroimage.2017.06.078
- Lancaster, J. L., Tordesillas-Gutiérrez, D., Martinez, M., Salinas, F., Evans, A., Zilles, K.,
  Mazziotta, J. C., & Fox, P. T. (2007). Bias between MNI and Talairach coordinates analyzed using
  the ICBM-152 brain template. *Human Brain Mapping, 28*(11), 1194–1205.
  https://doi.org/10.1002/hbm.20345
- Leuthardt, E. C., Schalk, G., Wolpaw, J. R., Ojemann, J. G., & Moran, D. W. (2004). A
  brain–computer interface using electrocorticographic signals in humans. *Journal of Neural
  Engineering, 1*(2), 63–71. https://doi.org/10.1088/1741-2560/1/2/001
- Manning, J. R., Jacobs, J., Fried, I., & Kahana, M. J. (2009). Broadband shifts in local field
  potential power spectra are correlated with single-neuron spiking in humans. *Journal of
  Neuroscience, 29*(43), 13613–13620. https://doi.org/10.1523/JNEUROSCI.2041-09.2009
- Miller, K. J. (2019). A library of human electrocorticographic data and analyses. *Nature Human
  Behaviour, 3*(11), 1225–1235. https://doi.org/10.1038/s41562-019-0678-3
- Miller, K. J., Hermes, D., Honey, C. J., Hebb, A. O., Ramsey, N. F., Knight, R. T., Ojemann, J. G.,
  & Fetz, E. E. (2012). Human motor cortical activity is selectively phase-entrained on underlying
  rhythms. *PLoS Computational Biology, 8*(9), e1002655. https://doi.org/10.1371/journal.pcbi.1002655
- Miller, K. J., Schalk, G., Fetz, E. E., den Nijs, M., Ojemann, J. G., & Rao, R. P. N. (2010).
  Cortical activity during motor execution, motor imagery, and imagery-based online feedback.
  *Proceedings of the National Academy of Sciences, 107*(9), 4430–4435.
  https://doi.org/10.1073/pnas.0913697107
- Miller, K. J., Zanos, S., Fetz, E. E., den Nijs, M., & Ojemann, J. G. (2009). Decoupling the
  cortical power spectrum reveals real-time representation of individual finger movements in
  humans. *Journal of Neuroscience, 29*(10), 3132–3137. https://doi.org/10.1523/JNEUROSCI.5506-08.2009
- Schalk, G., & Leuthardt, E. C. (2011). Brain-computer interfaces using electrocorticographic
  signals. *IEEE Reviews in Biomedical Engineering, 4*, 140–154.
  https://doi.org/10.1109/RBME.2011.2172408

## Apéndice (opcional): replicación grupal con los 7 sujetos

Hasta aquí trabajamos con un solo paciente y pocos electrodos, así que no podemos comparar nuestros
cocientes con los del artículo. La celda siguiente repite el cálculo de la sección 4 para **los 7
pacientes** del archivo y reúne todos sus electrodos significativos, como hicieron Miller et al.
(2010), que reportan **0.26** en HFB y **0.49** en LFB.

Es totalmente automática: ejecútala y espera unos 15 s. Antes de correrla, anota una predicción:
¿saldrán números parecidos a los del artículo? ¿Qué banda tendrá el cociente mayor?

In [ ]:
#@title 🌐 Replicación grupal: cociente imaginación : movimiento (7 sujetos)
import time


def subject_ratios(dat_real: dict, dat_imag: dict) -> tuple[np.ndarray, np.ndarray]:
    """Cociente imaginación:movimiento (HFB, LFB) en los electrodos significativos de un sujeto.

    Sigue a Miller et al. (2010): electrodos con cambio significativo en ambas bandas durante el
    movimiento (prueba t, Bonferroni sobre los electrodos del sujeto); mano y lengua combinadas.
    """
    scales = [np.asarray(d["scale_uv"], float) for d in (dat_real, dat_imag)]
    keep = np.all([s >= 0.2 * np.median(s) for s in scales], axis=0)   # descarta canales planos
    n_el = int(keep.sum())

    def psds(dat: dict):
        V = load_block(dat, keep)
        V -= V.mean(axis=1, keepdims=True)                              # CAR
        seg_m, seg_r, rest_from = trial_segments(dat["t_on"], dat["t_off"], len(V))
        f, p_m = segment_psds(V, seg_m, int(dat["srate"]))
        _, p_r = segment_psds(V, seg_r, int(dat["srate"]))
        return f, dat["stim_id"], p_m, p_r, dat["stim_id"][rest_from]

    f, st, pm, pr, st_rest = psds(dat_real)
    _, st_i, pmi, pri, st_rest_i = psds(dat_imag)
    out = {"HFB": [], "LFB": []}
    for code in (12, 11):                                               # mano, lengua
        mov, rest = pm[st == code], pr[st_rest == code]
        mov_i, rest_i = pmi[st_i == code], pri[st_rest_i == code]
        sig = np.ones(n_el, bool)
        for band in (HFB, LFB):
            _, pval = stats.ttest_ind(band_power(mov, f, band), band_power(rest, f, band), axis=0)
            sig &= pval < 0.05 / n_el
        for name, band in (("HFB", HFB), ("LFB", LFB)):
            real = band_power(mov, f, band).mean(0) - band_power(rest, f, band).mean(0)
            imag = band_power(mov_i, f, band).mean(0) - band_power(rest_i, f, band).mean(0)
            out[name].append((imag / real)[sig])
    return np.concatenate(out["HFB"]), np.concatenate(out["LFB"])


def gmean(r: np.ndarray) -> float:
    """Media geométrica de los cocientes positivos (los negativos no tienen logaritmo)."""
    r = r[r > 0]
    return float(np.exp(np.log(r).mean())) if r.size else np.nan


t0 = time.time()
per_subj = [subject_ratios(alldat[s][0], alldat[s][1]) for s in range(alldat.shape[0])]
print(f"Calculado en {time.time() - t0:.0f} s\n")
print(f"{'sujeto':<8}{'electrodos*':>12}{'GM HFB':>9}{'GM LFB':>9}")
for s, (h, l) in enumerate(per_subj):
    print(f"{s:<8}{len(h):>12}{gmean(h):>9.2f}{gmean(l):>9.2f}")
print("* pares electrodo-efector significativos (mano + lengua)\n")

H_all = np.concatenate([h for h, _ in per_subj])
L_all = np.concatenate([l for _, l in per_subj])
# Intervalo de confianza remuestreando SUJETOS (los electrodos de un mismo paciente no son independientes)
rng = np.random.default_rng(0)
picks = rng.integers(0, len(per_subj), (2000, len(per_subj)))
boot = np.array([[gmean(np.concatenate([per_subj[k][i] for k in pick])) for i in (0, 1)] for pick in picks])
ci = np.nanpercentile(boot, [2.5, 97.5], axis=0)
for i, (name, ref, r_all) in enumerate([("HFB", 0.26, H_all), ("LFB", 0.49, L_all)]):
    print(f"{name}: media geométrica = {gmean(r_all):.2f}   IC 95% por sujetos [{ci[0, i]:.2f}, {ci[1, i]:.2f}]"
          f"   (artículo: {ref})   razones > 0: {(r_all > 0).sum()}/{len(r_all)}")
n_gt = sum(gmean(l) > gmean(h) for h, l in per_subj)
print(f"LFB > HFB en {n_gt} de {len(per_subj)} sujetos")

fig, ax = plt.subplots(figsize=(6.5, 4.5))
palette = plt.cm.tab10(np.arange(len(per_subj)))
for i, (name, col, ref, r_all) in enumerate([("LFB", C_LFB, 0.49, L_all), ("HFB", C_HFB, 0.26, H_all)]):
    for s, rs in enumerate(per_subj):
        r = rs[1 if name == "LFB" else 0]
        jitter = np.random.default_rng(s).uniform(-0.2, 0.2, len(r))
        ax.scatter(np.full(len(r), i) + jitter, r, color=palette[s], s=16, alpha=0.8,
                   label=f"S{s}" if i == 0 else None)
    ax.hlines(gmean(r_all), i - 0.35, i + 0.35, color=col, lw=3)
    ax.hlines(ref, i - 0.35, i + 0.35, color="k", ls=":", lw=1.5)
ax.set(xticks=[0, 1], xticklabels=["LFB", "HFB"], ylabel="Imaginación : movimiento", ylim=(-0.5, 2.5),
       title="Línea gruesa: nuestra media · punteada: artículo")
ax.legend(ncol=4, fontsize=8, loc="upper right")
plt.show()

💡 **Para discutir.**
1. ¿Se conserva el orden LFB > HFB? ¿En cuántos sujetos?
2. Las magnitudes pueden diferir del artículo. ¿Por qué? Pistas: 7 pacientes en lugar de 8, la
   selección de electrodos, la definición exacta de "potencia normalizada" y la variabilidad entre
   pacientes (¿qué sujeto se desvía más?).
3. Los electrodos de un mismo paciente no son independientes. ¿Qué implica eso para el intervalo de
   confianza, y por qué lo calculamos remuestreando sujetos y no electrodos?